# 📱 AI_Phone: Hệ Thống RAG Tư Vấn Điện Thoại Toàn Diện (Kaggle Pipeline)

Pipeline hoàn chỉnh xử lý từ dữ liệu thô **PhoneDB** (`.html` / `.txt`), bóc tách dữ liệu sạch, chunking, tạo dataset fine-tune, huấn luyện Sentence Transformer trên GPU Kaggle, lập chỉ mục **FAISS + BM25**, và vận hành hệ thống RAG tư vấn (không cần LLM).

### 🔗 Liên kết:
- **Dataset nguồn (Raw PhoneDB)**: [https://www.kaggle.com/datasets/tngtrn23/phone-db](https://www.kaggle.com/datasets/tngtrn23/phone-db)
- **Thư mục lưu trữ đích (Google Drive)**: [https://drive.google.com/drive/folders/1i8dvXQ5WDQruGCLV-WQMQNzsA7EwtGXr](https://drive.google.com/drive/folders/1i8dvXQ5WDQruGCLV-WQMQNzsA7EwtGXr)
- **GitHub Codebase**: [https://github.com/tung-td06/AI_Phone.git](https://github.com/tung-td06/AI_Phone.git)

---

## ⚙️ 1. Kiểm tra Môi trường & GPU Kaggle
Khuyến nghị: Chọn **Accelerator: GPU T4 x2** hoặc **GPU P100** trong cài đặt Notebook (Settings -> Accelerator).

In [ ]:
import torch
import os
import sys
from pathlib import Path

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[!] Chú ý: Đang chạy trên CPU. Hãy bật GPU trong Settings -> Accelerator để tăng tốc!")

## 📦 2. Cài đặt Thư viện Phụ thuộc (Dependencies)

In [ ]:
!pip install -q faiss-cpu rank-bm25 sentence-transformers beautifulsoup4 lxml datasets accelerate

## 📥 3. Tải Mã Nguồn Dự Án từ GitHub

In [ ]:
%cd /kaggle/working
!rm -rf AI_Phone
!git clone https://github.com/tung-td06/AI_Phone.git
%cd /kaggle/working/AI_Phone
!ls -la

## 🔍 4. Xác minh Dataset Nguồn PhoneDB
Đảm bảo bạn đã bấm nút **+ Add Input** ở góc phải trên Kaggle và thêm dataset `tngtrn23/phone-db`.

In [ ]:
raw_input_dir = None
candidates = [
    "/kaggle/input/phone-db",
    "/kaggle/input/phone-db/data",
    "/kaggle/input/phone-db/database phone/data",
]
for c in candidates:
    if os.path.exists(c):
        raw_input_dir = c
        break

if raw_input_dir:
    print(f"[+] Đã tìm thấy dataset PhoneDB tại: {raw_input_dir}")
    files = [f for f in os.listdir(raw_input_dir) if not f.startswith('.')]
    print(f"[+] Số file/thư mục phát hiện ở root: {len(files):,} (Xem mẫu 5 file: {files[:5]})")
else:
    print("[!] Không tìm thấy dataset /kaggle/input/phone-db!")
    print("    Vui lòng bấm '+ Add Input' -> tìm 'tngtrn23/phone-db' để gắn vào notebook.")

## 🧹 5. BƯỚC 0: Tiền Xử Lý HTML Thô Sang Markdown Sạch (`xuli.py`)
Chuyển đổi dữ liệu HTML PhoneDB thành tài liệu Markdown sạch, bảo toàn 100% thuộc tính kỹ thuật và loại bỏ hoàn toàn mã HTML rác.

In [ ]:
cleaned_dir = "/kaggle/working/database_phone_cleaned"
os.makedirs(cleaned_dir, exist_ok=True)

!python xuli.py --input-dir "{raw_input_dir}" --output-dir "{cleaned_dir}" --workers 8

## 📑 6. BƯỚC 1 & BƯỚC 2: Kiểm Tra & Chunking Dữ Liệu (`B1.py` & `B2.py`)
Chia nhỏ thông số kỹ thuật theo 8 nhóm chuyên mục kỹ thuật chuẩn hóa:
1. Thông tin chung
2. Kích thước & Thiết kế (Dimensions)
3. Màn hình (Display)
4. Vi xử lý & Hiệu năng (Processor/Performance)
5. Bộ nhớ RAM/ROM (Memory)
6. Camera & Video
7. Pin & Sạc (Battery)
8. Kết nối mạng & Cảm biến (Connectivity)

In [ ]:
# Kiểm tra load dữ liệu qua B1
!python B1.py --dir "{cleaned_dir}" --limit 100

# Xuất file chunks jsonl qua B2
!python B2.py --dir "{cleaned_dir}" --output "/kaggle/working/chunks.jsonl"

## 🎯 7. BƯỚC 3: Chuẩn Bị Dữ Liệu Huấn Luyện (Triplets & Pairs) (`B3.py`)
Khai thác Hard Negatives và sinh bộ ba `(Anchor, Positive, Hard Negative)` để train mô hình embedding bằng loss MultipleNegativesRankingLoss (MNRL).

In [ ]:
finetune_dir = "/kaggle/working/dataset_finetune"
!python B3.py --dir "{cleaned_dir}" --output-dir "{finetune_dir}" --format both

## 🚀 8. BƯỚC 4: Huấn Luyện Fine-tune Embedding Model trên GPU (`B4.py`)
Huấn luyện mô hình `paraphrase-multilingual-MiniLM-L12-v2` chuyên biệt hóa cho lĩnh vực thông số điện thoại.

In [ ]:
model_output_dir = "/kaggle/working/final_phone_embedding_model"
checkpoint_dir = "/kaggle/working/checkpoints"

!python B4.py \
    --data-dir "{finetune_dir}" \
    --output-dir "{model_output_dir}" \
    --checkpoint-dir "{checkpoint_dir}" \
    --epochs 1 \
    --batch-size 32 \
    --lr 2e-5

## 🗄️ 9. BƯỚC 5: Xây Dựng CSDL Vector FAISS & BM25 Index (`B5.py`)
Mã hóa toàn bộ các chunk thông số bằng GPU thành vector dense 384 chiều và lưu vào FAISS index + BM25 sparse cache.

In [ ]:
rag_db_dir = "/kaggle/working/rag_db"

!python B5.py \
    --dir "{cleaned_dir}" \
    --model-dir "{model_output_dir}" \
    --rag-db-dir "{rag_db_dir}" \
    --batch-size 128

## 🏆 10. BƯỚC 7: Vận Hành Hệ Thống RAG & Chạy Evaluation Suite (`B7.py`)
Hệ thống RAG nâng cấp (P0 -> P3) hoạt động hoàn toàn độc lập, không cần LLM API:
- Bóc tách ngân sách thông minh (chống nhầm Galaxy A-15 với 15 triệu).
- Chấm điểm hiệu năng phần cứng thực tế theo bảng SoC benchmark chuẩn (`soc_database.py`).
- Chạy toàn bộ 16 ca kiểm định IR Metrics (Recall@K, Precision@K, nDCG@K).

In [ ]:
# Chạy bộ đánh giá 16 ca kiểm định
!python B7.py --rag-db-dir "{rag_db_dir}" --model-dir "{model_output_dir}" --eval

### Thử Nghiệm Một Số Truy Vấn Thực Tế

In [ ]:
# Case 1: Ngân sách 15 triệu
!python B7.py --rag-db-dir "{rag_db_dir}" --model-dir "{model_output_dir}" \
    --query "tai chinh 15 trieu mua dien thoai nao" --top-k 5

# Case 2: Điện thoại chơi game tốt nhất
!python B7.py --rag-db-dir "{rag_db_dir}" --model-dir "{model_output_dir}" \
    --query "May nao choi game tot nhat hien tai?" --top-k 5

# Case 3: Chơi PUBG dưới 10 triệu
!python B7.py --rag-db-dir "{rag_db_dir}" --model-dir "{model_output_dir}" \
    --query "Dien thoai choi PUBG tot duoi 10 trieu" --top-k 5

# Case 4: So sánh 2 máy
!python B7.py --rag-db-dir "{rag_db_dir}" --model-dir "{model_output_dir}" \
    --query "iPhone 15 Pro Max hay Xperia 1 V choi game tot hon?" --top-k 5

## 💾 11. BƯỚC 11: Đóng Gói File & Xuất Ra Google Drive (Không Cần Đăng Nhập Drive)

### 🌟 Quy trình thực hiện:
1. Chạy cell bên dưới: Kaggle sẽ tự động nén `rag_db` và `final_phone_embedding_model` thành file `phone_rag_data.zip`.
2. **Click trực tiếp vào link tải** xuất hiện ngay dưới cell (hoặc tải tại tab **Output** ở cột bên phải Kaggle).
3. Bấm vào link mở thư mục Google Drive: [Thư mục Google Drive](https://drive.google.com/drive/folders/1i8dvXQ5WDQruGCLV-WQMQNzsA7EwtGXr) và **kéo thả file `phone_rag_data.zip` vào** là hoàn tất 100%!

In [ ]:
import os
from IPython.display import FileLink, display, HTML

# 1. Chuyển vào /kaggle/working và nén các artifacts
%cd /kaggle/working
print("[*] Đang đóng gói CSDL Vector và Embedding Model vào phone_rag_data.zip...")
!zip -r -q phone_rag_data.zip AI_Phone/rag_db AI_Phone/final_phone_embedding_model

# 2. Kiểm tra dung lượng file đã nén
if os.path.exists('phone_rag_data.zip'):
    size_mb = os.path.getsize('phone_rag_data.zip') / (1024 * 1024)
    print(f"[+] ĐÃ ĐÓNG GÓI THÀNH CÔNG! Dung lượng file: {size_mb:.2f} MB")
    
    # 3. Hiển thị link tải 1-click trực tiếp trong Notebook
    print("\n" + "="*70)
    print("⬇️ BẤM VÀO ĐÂY ĐỂ TẢI FILE VỀ MÁY TÍNH:")
    display(FileLink('phone_rag_data.zip'))
    
    # 4. Hiển thị nút bấm tiện ích chuyển nhanh đến Google Drive
    drive_folder_url = "https://drive.google.com/drive/folders/1i8dvXQ5WDQruGCLV-WQMQNzsA7EwtGXr"
    html_btn = f'''
    <div style="margin-top: 15px; padding: 12px; background-color: #f0f7ff; border-left: 4px solid #1a73e8; border-radius: 4px;">
        <p style="margin: 0 0 8px 0; font-weight: bold; color: #1a73e8;">
            📁 BƯỚC TIẾP THEO: Kéo thả file vừa tải vào thư mục Google Drive của bạn:
        </p>
        <a href="{drive_folder_url}" target="_blank" style="display: inline-block; padding: 8px 16px; background-color: #1a73e8; color: white; text-decoration: none; border-radius: 4px; font-weight: bold;">
            🔗 Mở Thư Mục Google Drive (Click để mở)
        </a>
    </div>
    '''
    display(HTML(html_btn))
    print("="*70)
else:
    print("[!] Lỗi: Không tạo được file phone_rag_data.zip. Hãy kiểm tra lại các bước trước.")